# TAQ 1-minute prices — yearly parallel pull (CSV.gz)

Production variant of `legacy/download_hf_prices.ipynb`:

- **One file per year** in `data/wrds/high_freq_prices/hf_prices_<YYYY>.csv.gz`.
- **Parallel pulls** (`ThreadPoolExecutor`, one WRDS connection per thread, default 4 workers).
- **Streaming write** to gzipped CSV (one day at a time, no full-year buffer in RAM).
- **Atomic rename** (`.tmp` -> final) — a crashed run leaves no half-written gzip.
- **Resume safe** — years whose final file already exists are skipped.
- **Compact dtypes**: `permno` int32, `ticker`/`sym_root` category (in-memory), `price` rounded to 4 decimals, `time` as `HH:MM` (1-min grid: seconds always :00, dropped).

**Scale estimate** (full S&P 500, 6 years 2020-2025):
- ~614 PERMNO x ~1 508 trading days x 391 minutes = ~362 M rows
- CSV.gz: ~600-750 MB per year, ~3.5-4.5 GB total
- Wall time with 4 parallel workers: ~30-60 minutes (network bound)

In [ ]:
import wrds
import pandas as pd
import numpy as np
import gzip
import time
import logging
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

## 0) WRDS connection (main thread)

We open a primary connection here for the one-shot mapping pull below. Each worker thread will open its **own** connection (psycopg2 is not safe to share across threads).

In [ ]:
WRDS_USERNAME = 'aadmberrada'
db = wrds.Connection(wrds_username=WRDS_USERNAME)

## Configuration

In [ ]:
FIRST_DATE = '2021-01-01'
LAST_DATE  = '2025-12-31'

# Universe — pick (a) test or (b) full S&P 500

# (a) Test mode (10 names):
# PERMNO_LIST = [14593, 10107, 90319, 93436, 11703, 47896, 18411, 14541, 21936, 10104]

# (b) Full S&P 500 universe from the CRSP CIZ panel:
univ_df = pd.read_parquet('../data/wrds/sp500_constituents_daily.parquet')
PERMNO_LIST = univ_df['permno'].dropna().astype('int32').unique().tolist()

OUT_DIR = Path('../data/wrds/high_freq_prices')
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS       = list(range(int(FIRST_DATE[:4]), int(LAST_DATE[:4]) + 1))
MAX_WORKERS = 5   # parallel year-pulls; WRDS caps ~5-10 connections per user

print(f'Universe: {len(PERMNO_LIST):,} PERMNO  |  Years: {YEARS}  |  Workers: {MAX_WORKERS}')
print(f'Output:   {OUT_DIR}')

## 1) PERMNO -> sym_root/sym_suffix mapping (TAQMCLINK)

Pulled once on the main thread; the result is read-only and shared across workers.

In [ ]:
permno_tuple = tuple(int(p) for p in PERMNO_LIST)

taq_map = db.raw_sql(f"""
    SELECT DISTINCT permno,
                    sym_root,
                    COALESCE(sym_suffix, '') AS sym_suffix
    FROM wrdsapps.taqmclink
    WHERE permno IN {permno_tuple}
      AND sym_root IS NOT NULL
""")

# Compact dtypes: int32 for permno, category for symbol parts
taq_map = taq_map.astype({
    'permno':     'int32',
    'sym_root':   'category',
    'sym_suffix': 'category',
})

print(f"Universe PERMNO: {len(permno_tuple):,}")
print(f"(root, suffix) pairs: {len(taq_map):,}")
taq_map.head()

## 2) Per-day extraction functions (thread-safe)

All functions accept `db` as the first argument — no module-level connection. This way each worker thread can use its own connection.

Filters applied server-side (pushed to PostgreSQL `WHERE`):

- RTH only: `09:30:00 <= time_m <= 16:00:00`
- Real trade: `price > 0`
- Major exchanges: `ex IN ('N','P','T','Q','A','D')`
- No correction: `tr_corr = '00'`
- Sale conditions: keep blank or letters E/F (reject A,B,C,D,G..Z)
- Restrict to mapping universe via `(sym_root, sym_suffix) IN (...)`

In [ ]:
MEDIAN_CUTOFF = pd.Timestamp('2015-04-01')
GRID_TD       = pd.timedelta_range(start='09:30:00', end='16:00:00', freq='1min')   # 391 entries

# Build the (sym_root, sym_suffix) IN-list once (string used as-is in SQL)
pairs_sql = ', '.join(
    f"('{r}', '{s}')" for r, s in zip(taq_map['sym_root'].astype(str),
                                       taq_map['sym_suffix'].astype(str))
)


def fetch_day(db, yyyymmdd: str, pairs_sql_: str) -> pd.DataFrame:
    """Server-side filtered pull of one day's trades. time_m -> Timedelta."""
    df = db.raw_sql(f"""
        SELECT date,
               time_m,
               sym_root,
               COALESCE(sym_suffix, '') AS sym_suffix,
               price
        FROM taqmsec.ctm_{yyyymmdd}
        WHERE time_m BETWEEN TIME '09:30:00' AND TIME '16:00:00'
          AND price > 0
          AND ex IN ('N','P','T','Q','A','D')
          AND tr_corr = '00'
          AND (tr_scond IS NULL OR tr_scond = '' OR tr_scond !~ '[ABCDGHIJKLMNOPQRSTUVWXYZ]')
          AND (sym_root, COALESCE(sym_suffix, '')) IN ({pairs_sql_})
    """, date_cols=['date'])
    if not df.empty:
        df['time_m']     = pd.to_timedelta(df['time_m'].astype(str))
        df['sym_root']   = df['sym_root'].astype('category')
        df['sym_suffix'] = df['sym_suffix'].astype('category')
        df['price']      = df['price'].astype('float64')   # keep float64 for arithmetic safety
    return df


def previous_tick_minute_grid(trades: pd.DataFrame, day: pd.Timestamp) -> pd.DataFrame:
    """Reindex onto a 1-min grid 09:30..16:00 inclusive with previous-tick fill."""
    if trades.empty:
        return trades.iloc[0:0]
    out_chunks = []
    for (root, suffix), g in trades.groupby(['sym_root', 'sym_suffix'], sort=False, observed=True):
        g = g.sort_values('time_m')
        left = pd.DataFrame({'time_m': GRID_TD})
        merged = pd.merge_asof(
            left, g[['time_m', 'price']],
            on='time_m', direction='backward',
        )
        merged['date']       = day.date()
        merged['sym_root']   = root
        merged['sym_suffix'] = suffix
        # Render time as HH:MM string (1-min grid: seconds always :00, dropped for size)
        merged['time'] = (pd.Timestamp('1900-01-01') + merged['time_m']).dt.strftime('%H:%M')
        out_chunks.append(merged[['date', 'time', 'sym_root', 'sym_suffix', 'price']])
    return pd.concat(out_chunks, ignore_index=True)


def process_one_day(db, yyyymmdd: str, taq_map_local: pd.DataFrame, pairs_sql_: str) -> pd.DataFrame:
    day = pd.to_datetime(yyyymmdd, format='%Y%m%d')
    raw = fetch_day(db, yyyymmdd, pairs_sql_)

    if day < MEDIAN_CUTOFF and not raw.empty:
        raw = (raw.groupby(['sym_root', 'sym_suffix', 'time_m'], as_index=False, observed=True)
                  .agg(date=('date', 'first'), price=('price', 'median')))

    minute_grid = previous_tick_minute_grid(raw, day)
    if minute_grid.empty:
        return minute_grid

    minute_grid = minute_grid.merge(taq_map_local, on=['sym_root', 'sym_suffix'], how='left')
    minute_grid['ticker'] = (minute_grid['sym_root'].astype(str)
                             + minute_grid['sym_suffix'].astype(str))
    # Compact final dtypes + price rounding to 4 decimals
    minute_grid['permno'] = minute_grid['permno'].astype('Int32')
    minute_grid['price']  = minute_grid['price'].round(4)
    return minute_grid[['date', 'time', 'permno', 'ticker', 'price']]

## 3) Year worker + parallel runner

Each worker:
1. Skips silently if `hf_prices_<year>.csv.gz` already exists (resume safety).
2. Opens its own WRDS connection.
3. Lists `taqmsec.ctm_YYYYMMDD` tables for the year.
4. Streams each day's 1-min grid to a `.tmp` gzipped CSV.
5. Atomically renames the `.tmp` to the final filename on success.
6. Cleans up the `.tmp` and reports the error otherwise.

**Progress & log**:
- `tqdm.auto` shows an outer bar over years completed.
- Per-day timings + errors go to `logs/hf_prices_<timestamp>.log` (DEBUG level).
- Console output stays clean (only the bar + one summary line per year).
- `tail -f logs/hf_prices_*.log` from a terminal if you want a live per-day stream.

In [ ]:
# ---- Logging setup ----
log_dir = Path('../logs')
log_dir.mkdir(parents=True, exist_ok=True)
log_path = log_dir / f"hf_prices_{time.strftime('%Y%m%d_%H%M%S')}.log"

logger = logging.getLogger('hf_prices')
logger.setLevel(logging.DEBUG)
for h in list(logger.handlers):  # avoid duplicate handlers on cell re-run
    logger.removeHandler(h)

fh_log = logging.FileHandler(log_path, encoding='utf-8')
fh_log.setLevel(logging.DEBUG)
fh_log.setFormatter(logging.Formatter(
    '%(asctime)s [%(threadName)s] %(levelname)-7s %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
))
logger.addHandler(fh_log)

logger.info(
    f'=== HF prices pull starting | years={YEARS} '
    f'| universe={len(PERMNO_LIST)} | workers={MAX_WORKERS} ==='
)


# ---- Year worker ----
def process_year(year: int) -> dict:
    out_path = OUT_DIR / f'hf_prices_{year}.csv.gz'
    if out_path.exists():
        logger.info(f'year {year}: SKIP (already exists)')
        return dict(year=year, path=str(out_path), rows=0, days=0,
                    elapsed_s=0.0, status='skipped (already exists)')

    tmp_path = out_path.with_suffix(out_path.suffix + '.tmp')
    db_local = wrds.Connection(wrds_username=WRDS_USERNAME)
    try:
        ctm = db_local.raw_sql(f"""
            SELECT SUBSTRING(table_name FROM 5 FOR 8) AS yyyymmdd
            FROM information_schema.tables
            WHERE table_schema = 'taqmsec'
              AND table_name ~ '^ctm_[0-9]{{8}}$'
              AND SUBSTRING(table_name FROM 5 FOR 8) BETWEEN '{year}0101' AND '{year}1231'
            ORDER BY table_name
        """)
        if ctm.empty:
            logger.warning(f'year {year}: no CTM tables in range')
            return dict(year=year, path=None, rows=0, days=0, elapsed_s=0.0,
                        status='no CTM tables in range')

        logger.info(f'year {year}: starting {len(ctm)} days')
        n_rows, n_days = 0, 0
        t0_y = time.perf_counter()
        with gzip.open(tmp_path, 'wt', encoding='utf-8', newline='') as fhgz:
            first = True
            for yyyymmdd in ctm['yyyymmdd']:
                t0 = time.perf_counter()
                day_df = process_one_day(db_local, yyyymmdd, taq_map, pairs_sql)
                if day_df.empty:
                    logger.debug(f'year {year} day {yyyymmdd}: empty after filters')
                    continue
                day_df.to_csv(fhgz, header=first, index=False,
                              float_format='%.4f', mode='a')
                first   = False
                n_rows += len(day_df)
                n_days += 1
                logger.debug(
                    f'year {year} day {yyyymmdd}: {len(day_df):,} rows '
                    f'in {time.perf_counter() - t0:.1f}s'
                )
        tmp_path.rename(out_path)
        elapsed = time.perf_counter() - t0_y
        logger.info(
            f'year {year}: DONE {n_days} days, {n_rows:,} rows '
            f'in {elapsed/60:.1f} min'
        )
        return dict(year=year, path=str(out_path), rows=n_rows, days=n_days,
                    elapsed_s=elapsed, status='ok')
    except Exception as e:
        if tmp_path.exists():
            tmp_path.unlink()
        logger.exception(f'year {year}: failed')
        return dict(year=year, path=None, rows=0, days=0, elapsed_s=0.0,
                    status=f'ERROR {type(e).__name__}: {e}')
    finally:
        db_local.close()


# ---- Parallel runner with tqdm ----
t0_total = time.perf_counter()
results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(process_year, y): y for y in YEARS}
    with tqdm(total=len(futures), desc='Years', unit='yr') as pbar:
        for fut in as_completed(futures):
            r = fut.result()
            results.append(r)
            pbar.write(
                f"  year {r['year']}: {r['days']:>4} days, "
                f"{r['rows']:>10,} rows  [{r['elapsed_s']/60:5.1f} min]  "
                f"{r['status']}"
            )
            pbar.update(1)

elapsed_total = time.perf_counter() - t0_total
logger.info(f'=== HF prices pull complete in {elapsed_total/60:.1f} min ===')
print(f'\nDone in {elapsed_total/60:.1f} min')
print(f'Log:  {log_path}')
print(f'\nFiles in {OUT_DIR}:')
for p in sorted(OUT_DIR.glob('hf_prices_*.csv.gz')):
    print(f'  {p.name:30s}  {p.stat().st_size/1e6:>8.1f} MB')

## 4) Sanity checks (read multi-year CSV.gz back)

In [ ]:
files = sorted(OUT_DIR.glob('hf_prices_*.csv.gz'))
if not files:
    print('No output files found.')
else:
    # Read with explicit dtypes to keep memory low
    read_dtypes = {
        'permno': 'Int32',
        'ticker': 'category',
        'price':  'float64',
    }
    hf_prices = pd.concat(
        [pd.read_csv(f, parse_dates=['date'], dtype=read_dtypes) for f in files],
        ignore_index=True,
    )
    print(f'Loaded {len(files)} year file(s): {len(hf_prices):,} rows total\n')

    print(f"Memory usage: {hf_prices.memory_usage(deep=True).sum()/1e6:.1f} MB")
    print(f"Unique days:    {hf_prices['date'].nunique():>4}")
    print(f"Unique PERMNO:  {hf_prices['permno'].nunique():>4}")
    print(f"Unique tickers: {hf_prices['ticker'].nunique():>4}")
    n_nan = hf_prices['price'].isna().sum()
    print(f"Price NaN:      {n_nan:>10,}  ({n_nan/len(hf_prices)*100:.2f}%)")

    cnt = hf_prices.groupby(['date', 'permno'], observed=True).size()
    print(f"\nMinute counts per (date, permno) — should be 391:")
    print(f"  min={cnt.min()}  median={int(cnt.median())}  max={cnt.max()}")

    print(f'\nFirst 5 rows:')
    print(hf_prices.head().to_string(index=False))

## 5) Disconnect

In [ ]:
db.close()